# Reconstruction de surface 3D : Hoppe vs IGR

Ce notebook compare deux approches de reconstruction de surface à partir d'un nuage de points :

- **Hoppe (1992)** — plans tangents estimés par PCA, orientation globale via MST, extraction par marching cubes
- **IGR (Gropp et al. 2020)** — MLP qui apprend une SDF implicite avec contrainte eikonale

Les deux méthodes sont testées sur un nuage synthétique (SDF analytique) et sur trois nuages réels : Bunny, Egyptian Mask, Dragon.

In [1]:
import json
import math
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils as nn_utils
import trimesh
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial import cKDTree
from skimage import measure
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = 'notebook_connected'
Path('data').mkdir(exist_ok=True)
np.random.seed(0)

---
## 1  Méthode de Hoppe (1992)

Étapes :
1. Pour chaque point, estimer un **plan tangent** par PCA sur les k voisins les plus proches
2. Orienter les normales de façon cohérente avec un **arbre couvrant minimal (MST)**
3. Calculer une **grille de distances signées** (projection sur le plan tangent le plus proche)
4. Extraire l'iso-surface zéro par **marching cubes**

In [2]:
@dataclass
class TangentPlanes:
    centroids: np.ndarray
    normals:   np.ndarray


def estimate_tangent_planes(points, k):
    tree = cKDTree(points)
    _, idx = tree.query(points, k=k)
    neighborhoods = points[idx]
    centroids = neighborhoods.mean(axis=1)
    centered  = neighborhoods - centroids[:, None, :]
    cov = np.einsum('nki,nkj->nij', centered, centered) / k
    _, eigvecs = np.linalg.eigh(cov)
    normals = eigvecs[:, :, 0]
    normals /= np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12
    return TangentPlanes(centroids=centroids, normals=normals)


def orient_normals_mst(planes, k):
    centroids = planes.centroids
    normals   = planes.normals.copy()
    n = len(centroids)
    tree = cKDTree(centroids)
    _, idx = tree.query(centroids, k=k)
    rows, cols, weights = [], [], []
    for i in range(n):
        for j in idx[i, 1:]:
            w = 1.0 - abs(float(normals[i] @ normals[j]))
            rows.append(i); cols.append(int(j)); weights.append(w + 1e-9)
    graph = csr_matrix((weights, (rows, cols)), shape=(n, n))
    mst = (minimum_spanning_tree(graph) + minimum_spanning_tree(graph).T).tocsr()
    seed = int(np.argmax(centroids[:, 2]))
    if normals[seed, 2] < 0:
        normals[seed] = -normals[seed]
    visited = np.zeros(n, dtype=bool)
    visited[seed] = True
    stack = [seed]
    while stack:
        i = stack.pop()
        for j in mst.indices[mst.indptr[i]:mst.indptr[i + 1]]:
            j = int(j)
            if visited[j]:
                continue
            if normals[i] @ normals[j] < 0:
                normals[j] = -normals[j]
            visited[j] = True
            stack.append(j)
    return normals


def estimate_normals(points, k=32):
    planes = estimate_tangent_planes(points, k=k)
    return orient_normals_mst(planes, k=k)


def signed_distance_grid(planes, resolution, padding):
    centroids, normals = planes.centroids, planes.normals
    mn = centroids.min(axis=0) - padding
    mx = centroids.max(axis=0) + padding
    xs = np.linspace(mn[0], mx[0], resolution)
    ys = np.linspace(mn[1], mx[1], resolution)
    zs = np.linspace(mn[2], mx[2], resolution)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    grid = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    tree = cKDTree(centroids)
    _, nearest = tree.query(grid, k=1)
    f = np.einsum('ij,ij->i', grid - centroids[nearest], normals[nearest])
    return f.reshape(resolution, resolution, resolution), mn, mx


def reconstruct_hoppe(points, normals=None, k=32, resolution=160, padding=0.1):
    if normals is None:
        planes = estimate_tangent_planes(points, k=k)
        planes.normals = orient_normals_mst(planes, k=k)
    else:
        planes = TangentPlanes(centroids=points.copy(), normals=normals.copy())
    f, mn, mx = signed_distance_grid(planes, resolution=resolution, padding=padding)
    spacing = ((mx - mn) / (resolution - 1)).tolist()
    verts, faces, _, _ = measure.marching_cubes(f, level=0.0, spacing=spacing)
    verts += mn
    return trimesh.Trimesh(vertices=verts, faces=faces, process=True)

---
## 2  Réseau IGR (Gropp et al. 2020)

Un MLP de 8 couches (512 neurones, connexion *skip* à la couche 4) apprend une fonction implicite
$u : \mathbb{R}^3 \to \mathbb{R}$ en minimisant trois termes :

| Terme | Rôle |
|---|---|
| $\|u(x)\|$ sur les points de surface | L'isosurface zéro passe par les données |
| $\|\nabla u(x) - n(x)\|$ sur la surface | Les gradients suivent les normales |
| $(\|\nabla u(x)\| - 1)^2$ ailleurs | Contrainte eikonale (vraie SDF) |

In [3]:
class IGRNet(nn.Module):
    def __init__(self, d_in=3, d_hidden=512, n_layers=8, skip_layer=4,
                 beta=100.0, radius_init=0.5):
        super().__init__()
        self.d_in    = d_in
        self.skip_in = {skip_layer}
        dims = [d_in] + [d_hidden] * n_layers + [1]
        last = len(dims) - 2
        layers = []
        for i in range(len(dims) - 1):
            in_f, out_f = dims[i], dims[i + 1]
            if (i + 1) in self.skip_in:
                out_f -= d_in
            lin = nn.Linear(in_f, out_f)
            out_f_actual, in_f_actual = lin.weight.shape
            if i == last:
                nn.init.constant_(lin.weight, math.sqrt(math.pi) / math.sqrt(in_f_actual))
                nn.init.constant_(lin.bias, -radius_init)
            else:
                nn.init.normal_(lin.weight, 0.0, math.sqrt(2.0) / math.sqrt(out_f_actual))
                nn.init.constant_(lin.bias, 0.0)
                if i in self.skip_in:
                    lin.weight.data[:, -d_in:].zero_()
            layers.append(nn_utils.weight_norm(lin))
        self.layers     = nn.ModuleList(layers)
        self.activation = nn.Softplus(beta=beta)

    def forward(self, x):
        h = x
        for i, lin in enumerate(self.layers):
            if i in self.skip_in:
                h = torch.cat([h, x], dim=-1) / math.sqrt(2.0)
            h = lin(h)
            if i < len(self.layers) - 1:
                h = self.activation(h)
        return h


@dataclass
class LossTerms:
    value:   torch.Tensor
    normal:  torch.Tensor
    eikonal: torch.Tensor
    total:   torch.Tensor


def igr_loss(model, surface_pts, surface_normals, eik_pts, tau=1.0, lambda_eik=0.1):
    surface_pts = surface_pts.requires_grad_(True)
    u_s = model(surface_pts)
    value = u_s.abs().mean()
    grad_s = torch.autograd.grad(u_s, surface_pts, torch.ones_like(u_s),
                                  create_graph=True, retain_graph=True)[0]
    normal = (grad_s - surface_normals).norm(dim=-1).mean()
    eik_pts = eik_pts.requires_grad_(True)
    u_e = model(eik_pts)
    grad_e = torch.autograd.grad(u_e, eik_pts, torch.ones_like(u_e),
                                  create_graph=True, retain_graph=True)[0]
    eikonal = ((grad_e.norm(dim=-1) - 1.0) ** 2).mean()
    total = value + tau * normal + lambda_eik * eikonal
    return LossTerms(value, normal, eikonal, total)


def sample_eikonal_points(surface_pts, n_uniform, n_jitter, bbox=(-1.1, 1.1), sigma=0.1):
    device = surface_pts.device
    uniform = torch.empty(n_uniform, 3, device=device).uniform_(*bbox)
    if n_jitter == 0:
        return uniform
    idx    = torch.randint(0, surface_pts.shape[0], (n_jitter,), device=device)
    jitter = surface_pts[idx] + sigma * torch.randn(n_jitter, 3, device=device)
    return torch.cat([uniform, jitter], dim=0)


@dataclass
class Cloud:
    points:  np.ndarray
    normals: np.ndarray
    center:  np.ndarray
    scale:   float

    def denormalize(self, verts):
        return verts * self.scale + self.center


def normalize_cloud(points, normals):
    pts = np.asarray(points, dtype=np.float64)
    nrm = np.asarray(normals, dtype=np.float64)
    center = pts.mean(axis=0)
    centered = pts - center
    scale = float(np.abs(centered).max())
    pts_n = (centered / scale).astype(np.float32)
    nrm_n = (nrm / (np.linalg.norm(nrm, axis=1, keepdims=True) + 1e-12)).astype(np.float32)
    return Cloud(pts_n, nrm_n, center.astype(np.float32), scale)


def pick_device():
    if torch.backends.mps.is_available(): return torch.device('mps')
    if torch.cuda.is_available():         return torch.device('cuda')
    return torch.device('cpu')


def train_igr(model, cloud, n_iters=3000, batch_size=8192, tau=1.0, lambda_eik=0.1,
              lr=1e-4, log_every=100, loss_log_path=None, device=None):
    device = device or pick_device()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pts   = torch.from_numpy(cloud.points).to(device)
    nrm   = torch.from_numpy(cloud.normals).to(device)
    n_pts = pts.shape[0]
    history = {'iter': [], 'value': [], 'normal': [], 'eikonal': []}
    t0 = perf_counter()
    for it in range(1, n_iters + 1):
        idx     = torch.randint(0, n_pts, (batch_size,), device=device)
        surface = pts[idx]
        normals = nrm[idx]
        eik     = sample_eikonal_points(surface, n_uniform=batch_size, n_jitter=batch_size // 8)
        loss    = igr_loss(model, surface, normals, eik, tau=tau, lambda_eik=lambda_eik)
        optimizer.zero_grad(set_to_none=True)
        loss.total.backward()
        optimizer.step()
        if it % log_every == 0 or it == 1:
            v, n_l, e = loss.value.item(), loss.normal.item(), loss.eikonal.item()
            print(f'[{it:>5}/{n_iters}]  value={v:.3e}  normal={n_l:.3e}  eik={e:.3e}  total={loss.total.item():.3e}')
            history['iter'].append(it)
            history['value'].append(v)
            history['normal'].append(n_l)
            history['eikonal'].append(e)
    print(f'Entraînement terminé en {perf_counter()-t0:.1f}s')
    if loss_log_path:
        Path(loss_log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(loss_log_path).write_text(json.dumps(history, indent=2))
    return model


@torch.no_grad()
def evaluate_grid(model, resolution, bbox=(-1.1, 1.1), chunk=65536):
    device = next(model.parameters()).device
    xs = torch.linspace(bbox[0], bbox[1], resolution, device=device)
    gx, gy, gz = torch.meshgrid(xs, xs, xs, indexing='ij')
    grid = torch.stack([gx.ravel(), gy.ravel(), gz.ravel()], dim=1)
    out  = torch.empty(grid.shape[0], device=device)
    for i in range(0, grid.shape[0], chunk):
        out[i:i + chunk] = model(grid[i:i + chunk]).squeeze(-1)
    return out.reshape(resolution, resolution, resolution).cpu().numpy()


def extract_mesh_igr(model, resolution=256, bbox=(-1.1, 1.1)):
    field   = evaluate_grid(model, resolution, bbox=bbox)
    spacing = (bbox[1] - bbox[0]) / (resolution - 1)
    verts, faces, _, _ = measure.marching_cubes(field, level=0.0, spacing=(spacing,) * 3)
    verts += bbox[0]
    return trimesh.Trimesh(vertices=verts, faces=faces, process=True)


def chamfer_distance(points, mesh, n_samples=50_000):
    surf, _ = trimesh.sample.sample_surface(mesh, n_samples, seed=0)
    pc2mesh  = cKDTree(surf).query(points, k=1)[0].mean()
    mesh2pc  = cKDTree(points).query(surf,  k=1)[0].mean()
    return float((pc2mesh + mesh2pc) / 2)

---
## 3  Visualisation et pipeline

In [4]:
_SCENE = dict(
    camera=dict(eye=dict(x=0, y=0, z=2.2), up=dict(x=0, y=1, z=0)),
    xaxis=dict(showticklabels=False, title=''),
    yaxis=dict(showticklabels=False, title=''),
    zaxis=dict(showticklabels=False, title=''),
    bgcolor='rgb(15,15,25)')


def _mesh3d(mesh, color, name, max_faces=80_000):
    v, f = mesh.vertices, mesh.faces
    if len(f) > max_faces:
        f = f[np.random.default_rng(0).choice(len(f), max_faces, replace=False)]
    return go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2],
                     i=f[:,0], j=f[:,1], k=f[:,2],
                     color=color, name=name, showlegend=True,
                     lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3, roughness=0.6),
                     lightposition=dict(x=1, y=2, z=3))


def _scatter3d(pts, color, name, max_pts=15_000):
    if len(pts) > max_pts:
        pts = pts[np.random.default_rng(0).choice(len(pts), max_pts, replace=False)]
    return go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
                        marker=dict(size=1.5, color=color, opacity=0.7), name=name)


def visualize(pts, hoppe_mesh, igr_mesh, title=''):
    fig = make_subplots(rows=1, cols=3,
        specs=[[{'type': 'scene'}] * 3],
        subplot_titles=[
            f'Nuage — {title} ({len(pts):,} pts)',
            f'Hoppe  ({len(hoppe_mesh.vertices):,} v · {len(hoppe_mesh.faces):,} f)',
            f'IGR    ({len(igr_mesh.vertices):,} v · {len(igr_mesh.faces):,} f)',
        ],
        horizontal_spacing=0.02)
    fig.add_trace(_scatter3d(pts,        '#1f77b4', f'{title} (nuage)'), row=1, col=1)
    fig.add_trace(_mesh3d(hoppe_mesh,    '#ff7f0e', 'Hoppe'),            row=1, col=2)
    fig.add_trace(_mesh3d(igr_mesh,      '#2ca02c', 'IGR'),              row=1, col=3)
    fig.update_layout(
        title=dict(text=f'Reconstruction — {title} : Hoppe vs IGR', x=0.5,
                   font=dict(size=18, color='white')),
        paper_bgcolor='rgb(15,15,25)', font=dict(color='white'),
        scene=_SCENE, scene2=_SCENE, scene3=_SCENE,
        legend=dict(orientation='h', yanchor='bottom', y=0.01, xanchor='center', x=0.5,
                    bgcolor='rgba(30,30,50,0.7)'),
        margin=dict(l=0, r=0, t=70, b=30), height=640)
    return fig


def run_pipeline(ply_path, k=32, resolution=120, n_iters=2000):
    ply_path  = Path(ply_path)
    stem      = ply_path.stem
    out_hoppe = Path(f'data/{stem}_hoppe.ply')
    out_igr   = Path(f'data/{stem}_igr.ply')
    ckpt      = Path(f'data/{stem}_igr_checkpoint.pt')

    pts = np.array(trimesh.load(str(ply_path), process=False).vertices, dtype=np.float64)
    print(f'{stem}: {len(pts):,} points')

    if out_hoppe.exists():
        hoppe_mesh = trimesh.load(str(out_hoppe), process=False)
    else:
        t0 = perf_counter()
        hoppe_mesh = reconstruct_hoppe(pts, k=k, resolution=resolution, padding=0.05)
        hoppe_mesh.export(str(out_hoppe))
        print(f'Hoppe: {perf_counter()-t0:.1f}s — {len(hoppe_mesh.vertices):,} v, {len(hoppe_mesh.faces):,} f')

    if out_igr.exists():
        igr_mesh = trimesh.load(str(out_igr), process=False)
    else:
        normals = estimate_normals(pts, k=k)
        cloud   = normalize_cloud(pts, normals)
        model   = IGRNet()
        train_igr(model, cloud, n_iters=n_iters, batch_size=8192,
                  log_every=200, loss_log_path=f'data/{stem}_igr_loss.json')
        torch.save({'model_state': model.state_dict(),
                    'center': cloud.center, 'scale': cloud.scale}, str(ckpt))
        t0 = perf_counter()
        igr_mesh = extract_mesh_igr(model, resolution=128)
        igr_mesh.vertices = cloud.denormalize(igr_mesh.vertices)
        igr_mesh.export(str(out_igr))
        print(f'IGR: {perf_counter()-t0:.1f}s — {len(igr_mesh.vertices):,} v, {len(igr_mesh.faces):,} f')

    return pts, hoppe_mesh, igr_mesh

---
## 4  Nuage de points synthétique — première figure de test

La scène est composée d'un **cylindre** et d'un **cube**, tous deux percés de trois trous cylindriques.
La SDF est définie analytiquement ; on échantillonne la surface par *rejection sampling*.

In [5]:
CUBE_HALF  = 0.5
CYLINDER_R = 0.35
CYLINDER_H = 2.0
HOLE_R     = 0.30
HOLE_H     = 0.6


def sdf_cylinder(p, radius, half_height):
    d_xz = np.linalg.norm(p[:, [0, 2]], axis=1) - radius
    d_y  = np.abs(p[:, 1]) - half_height
    d    = np.column_stack([d_xz, d_y])
    return np.linalg.norm(np.maximum(d, 0.0), axis=1) + np.minimum(np.max(d, axis=1), 0.0)


def sdf_box(p, half_extents):
    q = np.abs(p) - np.asarray(half_extents)
    return np.linalg.norm(np.maximum(q, 0.0), axis=1) + np.minimum(np.max(q, axis=1), 0.0)


def sdf_scene(p):
    d_cyl    = sdf_cylinder(p, CYLINDER_R, CYLINDER_H)
    d_cube   = sdf_box(p, [CUBE_HALF] * 3)
    d_hole_x = sdf_cylinder(p[:, [1, 0, 2]], HOLE_R, HOLE_H)
    d_hole_y = sdf_cylinder(p, HOLE_R, HOLE_H)
    d_hole_z = sdf_cylinder(p[:, [0, 2, 1]], HOLE_R, HOLE_H)
    d_cube = np.maximum(d_cube, -d_hole_x)
    d_cube = np.maximum(d_cube, -d_hole_y)
    d_cube = np.maximum(d_cube, -d_hole_z)
    d_cyl  = np.maximum(d_cyl,  -d_hole_x)
    d_cyl  = np.maximum(d_cyl,  -d_hole_z)
    return np.minimum(d_cyl, d_cube)


def generate_point_cloud(n_samples=500_000, threshold=0.02, noise=0.005, seed=42):
    rng  = np.random.default_rng(seed)
    bbox = np.array([[-0.6, -2.1, -0.6], [0.6, 2.1, 0.6]])
    pts  = rng.uniform(bbox[0], bbox[1], (n_samples, 3)).astype(np.float32)
    sdf_vals = np.empty(n_samples, dtype=np.float32)
    for i in range(0, n_samples, 50_000):
        sdf_vals[i:i + 50_000] = sdf_scene(pts[i:i + 50_000])
    surface = pts[np.abs(sdf_vals) < threshold]
    surface += rng.normal(0, noise, surface.shape).astype(np.float32)
    return surface

In [6]:
sdf_pts = generate_point_cloud()
print(f'{len(sdf_pts):,} points générés')

# aperçu du nuage
idx = np.random.default_rng(1).choice(len(sdf_pts), 8_000, replace=False)
s   = sdf_pts[idx]
go.Figure(
    go.Scatter3d(x=s[:,0], y=s[:,1], z=s[:,2], mode='markers',
                 marker=dict(size=1.5, color=s[:,1], colorscale='Viridis', opacity=0.7)),
    layout=dict(title='Nuage SDF synthétique', height=500,
                paper_bgcolor='rgb(15,15,25)', font_color='white',
                scene=dict(bgcolor='rgb(15,15,25)'))
).show()

45,756 points générés


In [ ]:
out_h = Path('data/synthetic_hoppe.ply')
out_i = Path('data/synthetic_igr.ply')
ckpt  = Path('data/synthetic_igr_checkpoint.pt')

if out_h.exists():
    hoppe_synth = trimesh.load(str(out_h), process=False)
else:
    t0 = perf_counter()
    hoppe_synth = reconstruct_hoppe(sdf_pts, k=32, resolution=120, padding=0.1)
    hoppe_synth.export(str(out_h))
    print(f'Hoppe: {perf_counter()-t0:.1f}s — {len(hoppe_synth.vertices):,} v, {len(hoppe_synth.faces):,} f')

if out_i.exists():
    igr_synth = trimesh.load(str(out_i), process=False)
else:
    normals   = estimate_normals(sdf_pts, k=32)
    cloud     = normalize_cloud(sdf_pts, normals)
    model     = IGRNet()
    # CPU forcé : MPS sature la mémoire unifiée sur les petits nuages
    cpu = torch.device('cpu')
    train_igr(model, cloud, n_iters=800, batch_size=4096, log_every=200,
              loss_log_path='data/synthetic_igr_loss.json', device=cpu)
    torch.save({'model_state': model.state_dict(),
                'center': cloud.center, 'scale': cloud.scale}, str(ckpt))
    t0 = perf_counter()
    igr_synth = extract_mesh_igr(model, resolution=128)
    igr_synth.vertices = cloud.denormalize(igr_synth.vertices)
    igr_synth.export(str(out_i))
    print(f'IGR: {perf_counter()-t0:.1f}s — {len(igr_synth.vertices):,} v, {len(igr_synth.faces):,} f')

visualize(sdf_pts, hoppe_synth, igr_synth, 'Synthétique').show()

/opt/homebrew/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning:

`torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.



[    1/800]  value=8.421e-02  normal=9.172e-01  eik=2.796e-01  total=1.029e+00
[  200/800]  value=1.205e-02  normal=2.439e-01  eik=1.744e-02  total=2.577e-01


---
## 5  Données réelles

In [ ]:
pts_b, hoppe_b, igr_b = run_pipeline('data/bunny_L01.ply')
visualize(pts_b, hoppe_b, igr_b, title='Bunny').show()

In [ ]:
pts_m, hoppe_m, igr_m = run_pipeline('data/egyptian_mask.ply')
visualize(pts_m, hoppe_m, igr_m, title='Egyptian Mask').show()

In [ ]:
pts_d, hoppe_d, igr_d = run_pipeline('data/dragon.ply')
visualize(pts_d, hoppe_d, igr_d, title='Dragon').show()

---
## 6  Évaluation quantitative — Distance de Chamfer

La distance de Chamfer mesure l'écart symétrique entre le nuage original et la surface reconstruite :
$$d_C(P, M) = \frac{1}{2}\left(\frac{1}{|P|}\sum_{p \in P} \min_{m \in M}\|p-m\| + \frac{1}{|M|}\sum_{m \in M} \min_{p \in P}\|m-p\|\right)$$

In [ ]:
results = {}
for name, pts, h_mesh, i_mesh in [
    ('Bunny',         pts_b, hoppe_b, igr_b),
    ('Egyptian Mask', pts_m, hoppe_m, igr_m),
    ('Dragon',        pts_d, hoppe_d, igr_d),
]:
    cd_h = chamfer_distance(pts, h_mesh)
    cd_i = chamfer_distance(pts, i_mesh)
    results[name] = {'Hoppe': cd_h, 'IGR': cd_i}
    print(f'{name:15s}  Hoppe={cd_h:.4e}   IGR={cd_i:.4e}   delta={100*(cd_h-cd_i)/cd_h:+.1f}%')

fig = go.Figure()
names = list(results.keys())
for method, color in [('Hoppe', '#ff7f0e'), ('IGR', '#2ca02c')]:
    fig.add_bar(name=method, x=names,
                y=[results[n][method] for n in names],
                marker_color=color)
fig.update_layout(
    title='Distance de Chamfer (nuage ↔ mesh) — plus bas = meilleur',
    barmode='group', yaxis_title='Distance moyenne',
    paper_bgcolor='rgb(15,15,25)', plot_bgcolor='rgb(25,25,40)',
    font=dict(color='white'), height=420)
fig.show()

---
## 7  Convergence IGR

In [ ]:
stems  = ['bunny_L01', 'egyptian_mask', 'dragon']
titles = ['Bunny', 'Egyptian Mask', 'Dragon']
colors = {'value': '#1f77b4', 'normal': '#ff7f0e', 'eikonal': '#2ca02c'}
labels = {'value': 'Surface', 'normal': 'Normales', 'eikonal': 'Eikonale'}

fig = make_subplots(rows=1, cols=3, subplot_titles=titles)
for col, stem in enumerate(stems, 1):
    log = Path(f'data/{stem}_igr_loss.json')
    if not log.exists():
        continue
    h = json.loads(log.read_text())
    for key in ('value', 'normal', 'eikonal'):
        fig.add_trace(
            go.Scatter(x=h['iter'], y=h[key], name=labels[key],
                       line=dict(color=colors[key]), showlegend=(col == 1)),
            row=1, col=col)

fig.update_yaxes(type='log')
fig.update_layout(
    title='Évolution des termes de la loss IGR',
    paper_bgcolor='rgb(15,15,25)', plot_bgcolor='rgb(25,25,40)',
    font=dict(color='white'), height=380,
    legend=dict(bgcolor='rgba(30,30,50,0.7)'))
fig.show()